In [9]:
import numpy as np
import pywt
import cv2
import matplotlib.pyplot as plt

def compute_entropy(band):
    """Shannon entropy: -Σ p log2 p"""
    flattened = band.ravel()
    hist, _ = np.histogram(flattened, bins=256, range=(0.0, 1.0), density=False)
    prob = hist / np.sum(hist)  # 合計1に正規化
    prob = prob[prob > 0]       # log2(0) 回避
    return -np.sum(prob * np.log2(prob))


def extract_wavelet_stats_verbose(image, wavelet='db1', max_level=1, multiscale=True, visualize=False):
    """
    SWT（Stationary Wavelet Transform）で抽出した特徴量をラベル付きで出力
    - 統計: mean, std, energy, entropy
    - オプションで画像表示可能

    Parameters:
        image : ndarray (RGB or grayscale)
        wavelet : str
        max_level : int
        multiscale : bool
        visualize : bool

    Returns:
        features_dict : dict
    """
    # グレースケール化 & 正規化
    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    image = image.astype(np.float32) / 255.0

    coeffs = pywt.swt2(image, wavelet=wavelet, level=max_level)
    features_dict = {}

    levels = coeffs if multiscale else [coeffs[-1]]
    stat_names = ['mean', 'std', 'energy', 'entropy']
    band_names = ['cA', 'cH', 'cV', 'cD']

    for level_idx, (cA, (cH, cV, cD)) in enumerate(levels, start=1):
        bands = [cA, cH, cV, cD]
        for band_name, band in zip(band_names, bands):
            stats = [
                np.mean(band),
                np.std(band),
                np.sum(band**2),
                compute_entropy(band)
            ]
            for stat_name, stat_value in zip(stat_names, stats):
                key = f"L{level_idx}_{band_name}_{stat_name}"
                features_dict[key] = float(stat_value)

            # オプション表示
            if visualize:
                plt.figure()
                plt.imshow(band, cmap='ocean')
                plt.title(f"L{level_idx} {band_name}")
                plt.colorbar()
                plt.tight_layout()
                plt.show()

    return features_dict


In [ ]:
image = cv2.imread("sample.bmp")

features = extract_wavelet_stats_verbose(
    image,
    wavelet='db1',
    max_level=3,
    multiscale=True,
    visualize=True  # 各バンド画像を表示
)

print("▶ 特徴量（統計名付き）")
for name, value in features.items():
    print(f"{name:25s}: {value:.6f}")
